# 压缩空气储能仿真 Julia 版

本 Notebook 是 **先进绝热压缩空气储能 AA-CAES** 系统级 process 仿真的 Julia 语言版本。它与 Python 版保持同一套模型边界：多级压缩、级间冷却与压缩热回收、高压储气罐、TES 热储能、多级膨胀和发电输出。

这个版本的重点是观察 Julia 在科学计算表达上的写法。核心模型只使用 Julia Base，不强制依赖外部包；绘图单元会尝试使用 `Plots.jl`，如果当前环境未安装，会输出安装提示而不影响前面模型计算。


## 1. 工艺流程说明

流程链路：

```text
环境空气 -> 多级压缩机 -> 级间冷却 / TES 充热 -> 高压储气罐
       -> 静置保压与热损失
       -> TES 加热 -> 多级膨胀机 -> 发电机输出
```

运行阶段：

1. `charge`：压缩空气进入储气罐，压缩热进入 TES。
2. `hold`：储气罐和 TES 与环境换热。
3. `discharge`：高压空气经 TES 加热后膨胀发电。


## 2. 建模假设

1. 储气罐采用 0D 集中参数模型，不做三维空间分布。
2. 空气按理想气体处理，但比热容使用显式 `cp(T)` 变比热模型。
3. 压缩机和膨胀机采用多级等熵效率模型，各级等压比分配。
4. 换热器采用有效度模型，压缩热回收至 TES，放电时 TES 为膨胀前空气再热。
5. TES 采用集中参数模型，跟踪储热量和等效温度。
6. 时间推进采用显式欧拉法，便于代码可视化展示。
7. 管路压降、真实设备 map、三维 CFD 和经济性数据库不在当前 Julia 原型中实现。


## 3. 参数说明表

| 参数 | 含义 | 默认值 |
| --- | --- | --- |
| `compressor_stages` | 压缩机级数 | 3 |
| `expander_stages` | 膨胀机级数 | 2 |
| `mass_flow_kg_s` | 空气质量流量 | 12 kg/s |
| `storage_volume_m3` | 储气罐容积 | 1200 m3 |
| `min_storage_pressure_bar` | 最低放电压力 | 40 bar |
| `max_storage_pressure_bar` | 最高储气压力 | 120 bar |
| `heat_exchanger_effectiveness` | 换热器有效度 | 0.88 |
| `tes_mass_kg` | TES 储热介质质量 | 420000 kg |
| `charge_hours` / `hold_hours` / `discharge_hours` | 充电/静置/放电时间 | 4 / 2 / 4 h |
| `time_step_minutes` | 时间步长 | 2 min |


## 4. 数学模型

压缩机：

```text
T_out,s = T_in * (p_out / p_in)^((gamma - 1) / gamma)
h_out = h_in + (h_out,s - h_in) / eta_c
W_c = m_dot * (h_out - h_in)
```

储气罐：

```text
m_new = m_old + (m_in - m_out) dt
U_new = U_old + m_in h_in dt - m_out h_out dt + Q_wall dt
p_new = rho_new R T_new
```

TES：

```text
E_new = E_old + Q_charge dt - Q_discharge dt - UA_TES (T_TES - T_amb) dt
T_TES = T_amb + E_TES / (m_TES cp_TES)
```

膨胀机：

```text
T_out,s = T_in * (p_out / p_in)^((gamma - 1) / gamma)
h_out = h_in - eta_t * (h_in - h_out,s)
W_t = m_dot * (h_in - h_out)
```


## 5. 计算环境

核心计算只使用 Julia Base 和标准库 `Printf`。绘图在后面单独尝试加载 `Plots.jl`。


In [ ]:
# 计算环境
using Printf

function markdown_table(rows, columns)
    header = "| " * join(columns, " | ") * " |"
    separator = "| " * join(fill("---", length(columns)), " | ") * " |"
    body = String[]
    for row in rows
        push!(body, "| " * join([string(get(row, column, "")) for column in columns], " | ") * " |")
    end
    return join(vcat([header, separator], body), "\n")
end


## 6. 参数层代码

后续如果集成到系统，可以把这一层参数绑定到侧边栏。Julia 版变量命名与 Python 版保持一致，方便对比。


In [ ]:
# 参数层代码
operation_profile = "charge_hold_discharge"

ambient_temperature_c = 25.0
ambient_pressure_bar = 1.01325

compressor_stages = 3
expander_stages = 2
mass_flow_kg_s = 12.0
compressor_efficiency = 0.82
expander_efficiency = 0.86
motor_efficiency = 0.96
generator_efficiency = 0.96

heat_exchanger_effectiveness = 0.88
storage_volume_m3 = 1200.0
min_storage_pressure_bar = 40.0
max_storage_pressure_bar = 120.0
initial_storage_pressure_bar = 45.0
storage_heat_transfer_coefficient_wk = 2800.0

tes_mass_kg = 420000.0
tes_specific_heat_kj_kg_k = 0.92
tes_initial_temperature_c = 120.0
tes_ambient_loss_coefficient_wk = 1800.0
max_turbine_inlet_temperature_c = 520.0
minimum_tes_approach_temperature_k = 12.0

charge_hours = 4.0
hold_hours = 2.0
discharge_hours = 4.0
time_step_minutes = 2.0


## 7. 物性层代码

Julia 版采用显式 `cp(T)` 变比热模型。这样模型没有外部包依赖，也更适合展示物性计算如何进入压缩、膨胀和储气罐能量守恒。


In [ ]:
# 物性层代码
const R_AIR = 287.05
const T_REF = 298.15
const P_REF = 101325.0

ambient_temperature_k = ambient_temperature_c + 273.15
ambient_pressure_pa = ambient_pressure_bar * 1e5
tes_specific_heat_j_kg_k = tes_specific_heat_kj_kg_k * 1000.0

function cp_air_polynomial(T)
    theta = T - 300.0
    cp = 1006.0 + 0.085 * theta + 1.2e-4 * theta^2
    return max(cp, 950.0)
end

function h_air(T)
    x = T - 300.0
    x0 = 273.15 - 300.0
    return 1006.0 * (T - 273.15) + 0.085 * (x^2 - x0^2) / 2 + 1.2e-4 * (x^3 - x0^3) / 3
end

function T_from_h(h_target; low=180.0, high=1200.0)
    lo, hi = low, high
    for _ in 1:80
        mid = 0.5 * (lo + hi)
        if h_air(mid) < h_target
            lo = mid
        else
            hi = mid
        end
    end
    return 0.5 * (lo + hi)
end

function T_from_u_rho(u_target, rho; low=180.0, high=1200.0)
    lo, hi = low, high
    for _ in 1:80
        mid = 0.5 * (lo + hi)
        u_mid = h_air(mid) - R_AIR * mid
        if u_mid < u_target
            lo = mid
        else
            hi = mid
        end
    end
    return 0.5 * (lo + hi)
end

function air_props(T, p)
    cp = cp_air_polynomial(T)
    cv = cp - R_AIR
    h = h_air(T)
    rho = p / (R_AIR * T)
    s = cp * log(T / T_REF) - R_AIR * log(p / P_REF)
    u = h - R_AIR * T
    return Dict(:T => T, :p => p, :cp => cp, :cv => cv, :h => h, :u => u, :rho => rho, :s => s)
end

h_from_T_p(T, p) = air_props(T, p)[:h]
u_from_T_p(T, p) = air_props(T, p)[:u]
rho_from_T_p(T, p) = air_props(T, p)[:rho]

function T_p_from_u_rho(u, rho)
    T = T_from_u_rho(u, rho)
    p = rho * R_AIR * T
    return T, p
end

function isentropic_h_out(T_in, p_in, p_out)
    props = air_props(T_in, p_in)
    gamma = props[:cp] / props[:cv]
    T_out_s = T_in * (p_out / p_in)^((gamma - 1.0) / gamma)
    return h_air(T_out_s)
end

@printf("环境空气 cp = %.2f J/(kg*K)\n", air_props(298.15, 101325.0)[:cp])


## 8. 设备模型层代码

本层按设备边界拆分为六个小模型，代码中也用分段标题标出：

1. **单位换算与公共常量**：把参数层输入转成计算用 SI 单位。
2. **压缩机单级模型**：输入入口温度、入口压力、出口压力、质量流量和等熵效率，输出出口温度、出口焓和压缩功率。
3. **膨胀机单级模型**：输入入口温度、入口压力、出口压力、质量流量和等熵效率，输出出口温度、出口焓和膨胀功率。
4. **冷却器/换热器模型**：根据换热器有效度计算冷却后温度和换热量。
5. **定容储气罐动态模型**：根据质量守恒和能量守恒更新 `m_tank`、`T_tank` 和 `p_tank`。
6. **热储能 TES 动态模型**：根据充热、放热和热损失更新储热量和等效温度。


In [ ]:
# 设备模型层代码
# ============================================================
# 单位换算与公共常量
# ------------------------------------------------------------
# 作用：本 Julia 版在物性层已经完成 SI 单位换算。
# 后续设备函数统一使用 K、Pa、kg/s、J、W。
# ============================================================

# ============================================================
# 模型 1：压缩机单级模型 compressor_stage
# ------------------------------------------------------------
# 物理意义：把空气从 p_in 压缩到 p_out。
# 输入：T_in [K], p_in [Pa], p_out [Pa], eta_isentropic [-], mass_flow [kg/s]
# 输出：T_out [K], h_out [J/kg], power_w [W]
# 原理级公式：
#   T2 = T1 * (1 + (pi_c^((k-1)/k) - 1) / eta_c)
#   Wc = mdot * cp * (T2 - T1)
# 当前实现：用 cp(T) 变比热模型计算焓，再用焓差计算压缩功。
# ============================================================
function compressor_stage(T_in, p_in, p_out, eta_isentropic, mass_flow)
    h_in = h_from_T_p(T_in, p_in)
    h_out_s = isentropic_h_out(T_in, p_in, p_out)
    h_out = h_in + (h_out_s - h_in) / eta_isentropic
    T_out = T_from_h(h_out)
    power_w = mass_flow * (h_out - h_in)
    return T_out, h_out, power_w
end

# ============================================================
# 模型 2：膨胀机单级模型 expander_stage
# ------------------------------------------------------------
# 物理意义：高压热空气从 p_in 膨胀到 p_out，并输出轴功。
# 输入：T_in [K], p_in [Pa], p_out [Pa], eta_isentropic [-], mass_flow [kg/s]
# 输出：T_out [K], h_out [J/kg], power_w [W]
# 原理级公式：
#   T4 = T3 * (1 - eta_t * (1 - pi_t^((1-k)/k)))
#   Wt = mdot * cp * (T3 - T4)
# 当前实现：用等熵焓降乘以膨胀机等熵效率。
# ============================================================
function expander_stage(T_in, p_in, p_out, eta_isentropic, mass_flow)
    h_in = h_from_T_p(T_in, p_in)
    h_out_s = isentropic_h_out(T_in, p_in, p_out)
    h_out = h_in - eta_isentropic * (h_in - h_out_s)
    T_out = T_from_h(h_out)
    power_w = mass_flow * max(h_in - h_out, 0.0)
    return T_out, h_out, power_w
end

# ============================================================
# 模型 3：冷却器 / 换热器有效度模型 cooler
# ------------------------------------------------------------
# 物理意义：压缩后的高温空气向 TES 或冷端介质放热。
# 输入：热端入口温度、冷端入口温度、两侧热容率、换热器有效度。
# 输出：热端出口温度、冷端出口温度、换热功率。
# 原理级关系：Q = epsilon * C_min * (T_hot,in - T_cold,in)
# ============================================================
function cooler(T_hot_in, T_cold_in, heat_capacity_hot, heat_capacity_cold, effectiveness)
    delta_t = max(T_hot_in - T_cold_in, 0.0)
    if delta_t <= 0.0 || heat_capacity_hot <= 0.0 || heat_capacity_cold <= 0.0
        return T_hot_in, T_cold_in, 0.0
    end
    c_min = min(heat_capacity_hot, heat_capacity_cold)
    heat_w = effectiveness * c_min * delta_t
    T_hot_out = T_hot_in - heat_w / heat_capacity_hot
    T_cold_out = T_cold_in + heat_w / heat_capacity_cold
    return T_hot_out, T_cold_out, heat_w
end

# ============================================================
# 模型 4：定容储气罐动态模型 storage_tank_step
# ------------------------------------------------------------
# 物理意义：储气罐是定容容器，核心状态为 m_tank、T_tank、p_tank。
# 输入：上一时刻 state、进气质量流量、进气焓、出气质量流量、时间步长。
# 输出：下一时刻 state = Dict(:mass, :temperature, :pressure)
# 至少满足：
#   dm/dt = mdot_in - mdot_out
#   pV = mRT
# 当前实现：比等温模型更进一步，采用集中参数能量平衡：
#   U_new = U_old + mdot_in*h_in*dt - mdot_out*h_out*dt + Q_wall*dt
# ============================================================
function storage_tank_step(state, m_in, h_in, m_out, dt)
    m_old = max(state[:mass], 1e-6)
    T_old = state[:temperature]
    p_old = state[:pressure]
    u_old = u_from_T_p(T_old, p_old)
    h_out = h_from_T_p(T_old, p_old)
    heat_wall_w = -storage_heat_transfer_coefficient_wk * (T_old - ambient_temperature_k)

    internal_energy = m_old * u_old + m_in * h_in * dt - m_out * h_out * dt + heat_wall_w * dt
    m_new = max(m_old + (m_in - m_out) * dt, 1e-6)
    rho_new = max(m_new / storage_volume_m3, 1e-9)
    u_new = internal_energy / m_new
    T_new, p_new = T_p_from_u_rho(u_new, rho_new)
    return Dict(:mass => m_new, :temperature => T_new, :pressure => p_new)
end

# ============================================================
# 模型 5：热储能 TES 动态模型 tes_step
# ------------------------------------------------------------
# 物理意义：TES 储存压缩热，并在放电阶段加热进入膨胀机的空气。
# 输入：上一时刻 TES 状态、充热功率、放热功率、时间步长。
# 输出：下一时刻 TES 状态 = Dict(:energy_j, :temperature_k)
# 能量平衡：
#   E_new = E_old + Q_charge*dt - Q_discharge*dt - UA_loss*(T_TES - T_amb)*dt
# ============================================================
function tes_step(tes_state, heat_charge, heat_discharge, dt)
    energy_old = tes_state[:energy_j]
    temperature_old = tes_state[:temperature_k]
    loss_w = tes_ambient_loss_coefficient_wk * (temperature_old - ambient_temperature_k)
    energy_new = max(energy_old + heat_charge * dt - heat_discharge * dt - loss_w * dt, 0.0)
    temperature_k = ambient_temperature_k + energy_new / (tes_mass_kg * tes_specific_heat_j_kg_k)
    return Dict(:energy_j => energy_new, :temperature_k => temperature_k)
end

# ============================================================
# 辅助函数：多级压缩/膨胀的压力序列
# ------------------------------------------------------------
# 作用：在给定入口压力、出口压力和级数后，生成等压比分配的级间压力。
# ============================================================
function pressure_sequence(p_start, p_end, stages)
    ratio = (p_end / p_start)^(1.0 / stages)
    values = [p_start]
    for _ in 1:stages
        push!(values, values[end] * ratio)
    end
    values[end] = p_end
    return values
end


## 9. 系统状态机与求解层代码

这里执行完整充电、静置和放电周期，并输出所有后续绘图与结果分析所需的数组。


In [ ]:
# 系统状态机与求解层代码
function run_caes_cycle()
    dt = time_step_minutes * 60.0
    total_hours = charge_hours + hold_hours + discharge_hours
    total_steps = Int(round(total_hours * 3600.0 / dt)) + 1

    initial_pressure_pa = initial_storage_pressure_bar * 1e5
    initial_rho = rho_from_T_p(ambient_temperature_k, initial_pressure_pa)
    storage_state = Dict(
        :mass => initial_rho * storage_volume_m3,
        :temperature => ambient_temperature_k,
        :pressure => initial_pressure_pa
    )
    tes_state = Dict(
        :energy_j => max(tes_mass_kg * tes_specific_heat_j_kg_k * (tes_initial_temperature_c - ambient_temperature_c), 0.0),
        :temperature_k => tes_initial_temperature_c + 273.15
    )

    time_hours = Float64[]
    pressure_bar = Float64[]
    air_temperature_c = Float64[]
    air_mass_kg = Float64[]
    tes_temperature_c = Float64[]
    compressor_power_mw = Float64[]
    expander_power_mw = Float64[]
    net_power_mw = Float64[]
    charge_heat_mw = Float64[]
    discharge_heat_mw = Float64[]
    mode_by_step = String[]

    electric_input_j = 0.0
    electric_output_j = 0.0
    compression_heat_recovered_j = 0.0
    tes_heat_delivered_j = 0.0

    for step in 0:(total_steps - 1)
        current_hour = step * dt / 3600.0
        if current_hour < charge_hours
            mode = "charge"
        elseif current_hour < charge_hours + hold_hours
            mode = "hold"
        else
            mode = "discharge"
        end

        p_tank = storage_state[:pressure]
        T_tank = storage_state[:temperature]
        m_dot_in = 0.0
        m_dot_out = 0.0
        h_in_storage = 0.0
        compressor_power_w = 0.0
        expander_power_w = 0.0
        heat_charge_w = 0.0
        heat_discharge_w = 0.0

        if mode == "charge"
            if p_tank >= 0.98 * max_storage_pressure_bar * 1e5
                mode = "charge_pressure_limited"
            else
                m_dot_in = mass_flow_kg_s
                p_final = min(max(p_tank * 1.005, min_storage_pressure_bar * 1e5), 0.98 * max_storage_pressure_bar * 1e5)
                pressures = pressure_sequence(ambient_pressure_pa, p_final, compressor_stages)
                T_air = ambient_temperature_k
                tes_capacity_rate = max(tes_mass_kg * tes_specific_heat_j_kg_k / dt, 1.0)

                for stage in 1:compressor_stages
                    T_air, h_air_stage, power_w = compressor_stage(
                        T_air,
                        pressures[stage],
                        pressures[stage + 1],
                        compressor_efficiency,
                        m_dot_in
                    )
                    compressor_power_w += power_w / motor_efficiency

                    cp_hot = air_props(T_air, pressures[stage + 1])[:cp]
                    hot_capacity_rate = m_dot_in * cp_hot
                    T_air, _, q_w = cooler(
                        T_air,
                        tes_state[:temperature_k],
                        hot_capacity_rate,
                        tes_capacity_rate,
                        heat_exchanger_effectiveness
                    )
                    heat_charge_w += q_w
                end

                h_in_storage = h_from_T_p(T_air, p_final)
                compression_heat_recovered_j += heat_charge_w * dt
                electric_input_j += compressor_power_w * dt
            end

        elseif mode == "discharge"
            if p_tank <= 1.05 * min_storage_pressure_bar * 1e5 || storage_state[:mass] <= 1e-6
                mode = "discharge_pressure_limited"
            else
                available_mass_flow = max((storage_state[:mass] - 1e-6) / dt, 0.0)
                m_dot_out = min(mass_flow_kg_s, available_mass_flow)
                T_air = T_tank
                p_start = p_tank
                p_final = ambient_pressure_pa

                cp_current = air_props(T_air, p_start)[:cp]
                target_turbine_inlet_k = min(
                    max_turbine_inlet_temperature_c + 273.15,
                    max(tes_state[:temperature_k] - minimum_tes_approach_temperature_k, T_air)
                )
                preheat_w = max(m_dot_out * cp_current * (target_turbine_inlet_k - T_air), 0.0)
                available_tes_power = max(tes_state[:energy_j] / dt, 0.0)
                preheat_w = min(preheat_w, available_tes_power)
                if preheat_w > 0.0
                    T_air += preheat_w / max(m_dot_out * cp_current, 1e-9)
                    heat_discharge_w += preheat_w
                end

                pressures = pressure_sequence(p_start, p_final, expander_stages)
                for stage in 1:expander_stages
                    T_air, h_air_stage, power_w = expander_stage(
                        T_air,
                        pressures[stage],
                        pressures[stage + 1],
                        expander_efficiency,
                        m_dot_out
                    )
                    expander_power_w += power_w * generator_efficiency

                    if stage < expander_stages
                        cp_reheat = air_props(T_air, pressures[stage + 1])[:cp]
                        target_reheat_k = min(
                            max_turbine_inlet_temperature_c + 273.15,
                            max(tes_state[:temperature_k] - minimum_tes_approach_temperature_k, T_air)
                        )
                        reheat_w = max(m_dot_out * cp_reheat * (target_reheat_k - T_air), 0.0)
                        available_tes_power = max((tes_state[:energy_j] - heat_discharge_w * dt) / dt, 0.0)
                        reheat_w = min(reheat_w, available_tes_power)
                        if reheat_w > 0.0
                            T_air += reheat_w / max(m_dot_out * cp_reheat, 1e-9)
                            heat_discharge_w += reheat_w
                        end
                    end
                end

                electric_output_j += expander_power_w * dt
                tes_heat_delivered_j += heat_discharge_w * dt
            end
        end

        if startswith(mode, "charge")
            storage_state = storage_tank_step(storage_state, m_dot_in, h_in_storage, 0.0, dt)
            tes_state = tes_step(tes_state, heat_charge_w, 0.0, dt)
        elseif startswith(mode, "discharge")
            storage_state = storage_tank_step(storage_state, 0.0, 0.0, m_dot_out, dt)
            tes_state = tes_step(tes_state, 0.0, heat_discharge_w, dt)
        else
            storage_state = storage_tank_step(storage_state, 0.0, 0.0, 0.0, dt)
            tes_state = tes_step(tes_state, 0.0, 0.0, dt)
        end

        push!(time_hours, current_hour)
        push!(pressure_bar, storage_state[:pressure] / 1e5)
        push!(air_temperature_c, storage_state[:temperature] - 273.15)
        push!(air_mass_kg, storage_state[:mass])
        push!(tes_temperature_c, tes_state[:temperature_k] - 273.15)
        push!(compressor_power_mw, compressor_power_w / 1e6)
        push!(expander_power_mw, expander_power_w / 1e6)
        push!(net_power_mw, (expander_power_w - compressor_power_w) / 1e6)
        push!(charge_heat_mw, heat_charge_w / 1e6)
        push!(discharge_heat_mw, heat_discharge_w / 1e6)
        push!(mode_by_step, mode)
    end

    round_trip_efficiency = electric_input_j > 0.0 ? electric_output_j / electric_input_j : 0.0
    return Dict(
        :time_hours => time_hours,
        :pressure_bar => pressure_bar,
        :air_temperature_c => air_temperature_c,
        :air_mass_kg => air_mass_kg,
        :tes_temperature_c => tes_temperature_c,
        :compressor_power_mw => compressor_power_mw,
        :expander_power_mw => expander_power_mw,
        :net_power_mw => net_power_mw,
        :charge_heat_mw => charge_heat_mw,
        :discharge_heat_mw => discharge_heat_mw,
        :mode_by_step => mode_by_step,
        :electric_input_j => electric_input_j,
        :electric_output_j => electric_output_j,
        :compression_heat_recovered_j => compression_heat_recovered_j,
        :tes_heat_delivered_j => tes_heat_delivered_j,
        :round_trip_efficiency => round_trip_efficiency
    )
end

results = run_caes_cycle()

time_hours = results[:time_hours]
pressure_bar = results[:pressure_bar]
air_temperature_c = results[:air_temperature_c]
air_mass_kg = results[:air_mass_kg]
tes_temperature_c = results[:tes_temperature_c]
compressor_power_mw = results[:compressor_power_mw]
expander_power_mw = results[:expander_power_mw]
net_power_mw = results[:net_power_mw]
charge_heat_mw = results[:charge_heat_mw]
discharge_heat_mw = results[:discharge_heat_mw]
mode_by_step = results[:mode_by_step]
round_trip_efficiency = results[:round_trip_efficiency]

@printf("仿真步数: %d\n", length(time_hours))
@printf("最高储气压力: %.2f bar\n", maximum(pressure_bar))
@printf("最低储气压力: %.2f bar\n", minimum(pressure_bar))
@printf("往返效率: %.2f %%\n", round_trip_efficiency * 100)


## 10. 结果可视化代码

该单元尝试使用 `Plots.jl`。如果环境未安装 `Plots`，模型计算结果仍然可用，只是不会绘制图像。


In [ ]:
# 结果可视化代码
try
    @eval using Plots
    mode_order = ["charge", "charge_pressure_limited", "hold", "discharge", "discharge_pressure_limited"]
    mode_to_value = Dict(mode => index for (index, mode) in enumerate(mode_order))
    mode_values = [get(mode_to_value, mode, 0) for mode in mode_by_step]

    p1 = plot(time_hours, pressure_bar, linewidth=2, label="储气压力", xlabel="时间 / h", ylabel="bar", title="储气罐压力动态")
    hline!(p1, [max_storage_pressure_bar], linestyle=:dash, label="最高压力")
    hline!(p1, [min_storage_pressure_bar], linestyle=:dash, label="最低压力")

    p2 = plot(time_hours, air_temperature_c, linewidth=2, label="空气温度", xlabel="时间 / h", ylabel="degC", title="储气罐空气温度")
    hline!(p2, [ambient_temperature_c], linestyle=:dash, label="环境温度")

    p3 = plot(time_hours, air_mass_kg, linewidth=2, label="空气质量", xlabel="时间 / h", ylabel="kg", title="储气罐空气质量")

    p4 = plot(time_hours, tes_temperature_c, linewidth=2, label="TES 温度", xlabel="时间 / h", ylabel="degC", title="TES 等效温度")

    p5 = plot(time_hours, compressor_power_mw, linewidth=2, label="压缩耗电", xlabel="时间 / h", ylabel="MW", title="系统功率动态")
    plot!(p5, time_hours, expander_power_mw, linewidth=2, label="膨胀发电")
    plot!(p5, time_hours, net_power_mw, linewidth=2, label="净功率")

    p6 = plot(time_hours, mode_values, seriestype=:steppost, linewidth=2, label="运行模式", xlabel="时间 / h", title="运行模式时间轴")
    yticks!(p6, collect(values(mode_to_value)), collect(keys(mode_to_value)))

    plot(p1, p2, p3, p4, p5, p6, layout=(3, 2), size=(1200, 900))
catch err
    println("当前 Julia 环境未安装或无法加载 Plots.jl，因此跳过绘图。")
    println("如需绘图，可在 Julia 中运行: using Pkg; Pkg.add(\"Plots\")")
    println("模型结果数组仍已生成，可查看 pressure_bar、tes_temperature_c、net_power_mw 等变量。")
end


## 11. 关键结果输出

这里集中输出输入电量、输出电量、往返效率、压力范围、TES 温度和热量利用。


In [ ]:
# 关键结果输出
electric_input_mwh = results[:electric_input_j] / 3.6e9
electric_output_mwh = results[:electric_output_j] / 3.6e9
compression_heat_recovered_mwh = results[:compression_heat_recovered_j] / 3.6e9
tes_heat_delivered_mwh = results[:tes_heat_delivered_j] / 3.6e9

summary_rows = [
    Dict("指标" => "压缩阶段电输入", "数值" => @sprintf("%.3f", electric_input_mwh), "单位" => "MWh"),
    Dict("指标" => "膨胀阶段电输出", "数值" => @sprintf("%.3f", electric_output_mwh), "单位" => "MWh"),
    Dict("指标" => "系统往返效率", "数值" => @sprintf("%.2f", round_trip_efficiency * 100), "单位" => "%"),
    Dict("指标" => "最高储气压力", "数值" => @sprintf("%.2f", maximum(pressure_bar)), "单位" => "bar"),
    Dict("指标" => "最低储气压力", "数值" => @sprintf("%.2f", minimum(pressure_bar)), "单位" => "bar"),
    Dict("指标" => "最终储气压力", "数值" => @sprintf("%.2f", pressure_bar[end]), "单位" => "bar"),
    Dict("指标" => "最高储气温度", "数值" => @sprintf("%.2f", maximum(air_temperature_c)), "单位" => "degC"),
    Dict("指标" => "最终 TES 温度", "数值" => @sprintf("%.2f", tes_temperature_c[end]), "单位" => "degC"),
    Dict("指标" => "压缩热回收量", "数值" => @sprintf("%.3f", compression_heat_recovered_mwh), "单位" => "MWh_th"),
    Dict("指标" => "TES 放热量", "数值" => @sprintf("%.3f", tes_heat_delivered_mwh), "单位" => "MWh_th")
]

println(markdown_table(summary_rows, ["指标", "数值", "单位"]))

mode_counts = Dict(mode => count(==(mode), mode_by_step) for mode in unique(mode_by_step))
println("运行模式步数: ", mode_counts)


## 12. 结果分析提示

1. 调大 `storage_volume_m3` 后，储气压力变化更慢，系统可放电时间更长。
2. 调高 `heat_exchanger_effectiveness` 后，TES 能回收更多压缩热，放电阶段再热能力增强。
3. 调大 `tes_mass_kg` 后，TES 温度变化更平缓，但温升也会降低。
4. Julia 版采用显式 `cp(T)` 模型，优点是依赖少、代码透明；若后续追求更高物性精度，可以接入 CoolProp 包装或其他物性库。
5. 与 Python 版相比，Julia 版更适合继续扩展到更大规模参数扫描或微分方程求解，但当前论文原型更重要的是模型透明和代码可视化。
